# Update sourced skills

Checks each **externally-sourced** skill in this repo against its upstream
repository, shows a change-list (with diffs) when an update exists, and lets you
**apply** or **disregard** each one.

The list of sources lives in [`skill-plugin-sources.json`](skill-plugin-sources.json). Both top-level skill folders and newly-added external plugin directories are tracked there now. Skills listed under `excluded` are never touched — currently:

- **caveman** — locally modified to suit personal use.
- **pbi-visual-rendering**, **semantic-modeling-prepforai** — my own skills.

**How it works (stdlib only — needs `git` on PATH):**

1. *Setup* — load the manifest.
2. *Step 1 — Check* — shallow-clone each upstream repo and diff its subpath
   against the local folder.
3. *Step 2 — Decide* — edit the `DECISIONS` dict (`apply` / `skip`).
4. *Step 3 — Apply* — copy chosen upstream files in. Local-only files (e.g. the
   `LICENSE` copies) are **never deleted**; `last_synced_sha` is stamped back
   into the manifest.

Review with `git diff` and commit when satisfied.

In [ ]:
import json, subprocess, tempfile, shutil, difflib
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "skill-plugin-sources.json").is_file():
            return p
    raise FileNotFoundError("skill-plugin-sources.json not found in CWD or any parent. "
                            "Run this notebook from inside the skills repo.")

REPO_ROOT = find_repo_root(Path.cwd())
MANIFEST_PATH = REPO_ROOT / "skill-plugin-sources.json"
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
IGNORE = set(manifest.get("ignore", []))
SKILLS = manifest["skills"]

print(f"Repo root: {REPO_ROOT}")
print(f"\nTracking {len(SKILLS)} sourced skill(s):")
for s in SKILLS:
    print(f"  - {s['name']:<32} <- {s['repo']} :: {s['subpath'] or '(repo root)'}")
print("\nExcluded (never auto-updated):")
for e in manifest.get("excluded", []):
    print(f"  - {e['name']:<32} {e['reason']}")

In [ ]:
def _run(args, cwd=None):
    res = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(f"command failed: {' '.join(args)}\n{res.stderr.strip()}")
    return res.stdout.strip()

_clone_cache = {}
def clone_upstream(repo, branch):
    # Shallow-clone a repo once per session; returns (path, head_sha).
    key = (repo, branch)
    if key not in _clone_cache:
        dest = Path(tempfile.mkdtemp(prefix="skillsrc_"))
        _run(["git", "clone", "--depth", "1", "--branch", branch, repo, str(dest)])
        sha = _run(["git", "rev-parse", "HEAD"], cwd=dest)
        _clone_cache[key] = (dest, sha)
    return _clone_cache[key]

def _ignored(rel: Path):
    return any(part in IGNORE for part in rel.parts)

def _list_files(root: Path):
    out = {}
    if root.is_dir():
        for p in root.rglob("*"):
            if p.is_file():
                rel = p.relative_to(root)
                if not _ignored(rel):
                    out[rel.as_posix()] = p
    return out

def scan_skill(skill):
    # Compare upstream subpath -> local folder; return a change report.
    up_root, head_sha = clone_upstream(skill["repo"], skill["branch"])
    up_dir = (up_root / skill["subpath"]) if skill["subpath"] else up_root
    local_dir = REPO_ROOT / skill["local"]
    up_files, local_files = _list_files(up_dir), _list_files(local_dir)

    added, modified, local_only = [], [], []
    for rel, up_path in sorted(up_files.items()):
        lp = local_files.get(rel)
        if lp is None:
            added.append(rel)
        elif up_path.read_bytes() != lp.read_bytes():
            modified.append(rel)
    for rel in sorted(local_files):
        if rel not in up_files:
            local_only.append(rel)

    return {"skill": skill, "head_sha": head_sha, "up_files": up_files,
            "local_dir": local_dir, "added": added, "modified": modified,
            "local_only": local_only, "has_update": bool(added or modified)}

def unified_diff(up_path: Path, local_path: Path, rel: str):
    try:
        up = up_path.read_text(encoding="utf-8").splitlines(keepends=True)
        lo = local_path.read_text(encoding="utf-8").splitlines(keepends=True)
    except UnicodeDecodeError:
        return f"(binary file '{rel}' differs)\n"
    return "".join(difflib.unified_diff(lo, up, fromfile=f"local/{rel}", tofile=f"upstream/{rel}"))

## Step 1 — Check for updates

Set `SHOW_DIFFS = False` for a summary-only view.

In [ ]:
SHOW_DIFFS = True

RESULTS = {}
for skill in SKILLS:
    name = skill["name"]
    print("=" * 78)
    print(f"{name}   (last synced: {skill.get('last_synced_sha') or 'never'})")
    try:
        rep = scan_skill(skill)
    except Exception as e:
        print(f"  !! could not check: {e}")
        continue
    RESULTS[name] = rep
    print(f"  upstream HEAD: {rep['head_sha']}")
    if not rep["has_update"]:
        print("  up to date — no changes to apply.")
    else:
        for r in rep["added"]:
            print(f"      + (new)      {r}")
        for r in rep["modified"]:
            print(f"      ~ (modified) {r}")
    if rep["local_only"]:
        print(f"  local-only (kept, never deleted): {', '.join(rep['local_only'])}")
    if SHOW_DIFFS and rep["modified"]:
        for r in rep["modified"]:
            print("-" * 70)
            print(unified_diff(rep["up_files"][r], rep["local_dir"] / r, r))

print("=" * 78)
updatable = [n for n, r in RESULTS.items() if r["has_update"]]
print(f"Skills with updates available: {updatable or 'none'}")

## Step 2 — Decide

Running the next cell builds `DECISIONS` with every updatable skill defaulted to
`"skip"`. **Edit the values to `"apply"`** for the ones you want, then run Step 3.
(Re-running this cell resets your choices.)

In [ ]:
DECISIONS = {name: "skip" for name, rep in RESULTS.items() if rep["has_update"]}

if not DECISIONS:
    print("Nothing to decide — all sourced skills are up to date.")
else:
    print("DECISIONS — change 'skip' to 'apply' where you want the update:\n")
    print("DECISIONS = {")
    for k in DECISIONS:
        print(f'    "{k}": "skip",')
    print("}")
    print("\nEdit the dict above (in this cell), then run Step 3.")

## Step 3 — Apply

Copies upstream `added` + `modified` files into each chosen local folder,
preserves local-only files, and stamps `last_synced_sha` into the manifest.

In [ ]:
def apply_update(rep):
    copied = []
    for rel in rep["added"] + rep["modified"]:
        dst = rep["local_dir"] / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(rep["up_files"][rel], dst)
        copied.append(rel)
    for s in manifest["skills"]:
        if s["name"] == rep["skill"]["name"]:
            s["last_synced_sha"] = rep["head_sha"]
    return copied

applied_any = False
for name, decision in DECISIONS.items():
    if str(decision).strip().lower() != "apply":
        print(f"skip   {name}")
        continue
    rep = RESULTS.get(name)
    if not rep or not rep["has_update"]:
        print(f"skip   {name} (nothing to apply)")
        continue
    copied = apply_update(rep)
    applied_any = True
    print(f"APPLY  {name}: wrote {len(copied)} file(s):")
    for r in copied:
        print(f"         {r}")

if applied_any:
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    print(f"\nStamped {MANIFEST_PATH.name} with new last_synced_sha values.")
    print("Next: review with `git diff`, then commit. Local-only files were preserved.")
else:
    print("\nNo updates applied.")